In [ ]:
#1,Quality control -remove blanks
# feb 2, 2026 i have used R package Decontam to statistically remove the blank

In [ ]:
#THIS CELL BREAKS THE TAXONOMY TABLE INTO DOMAIN-SPECIES
import pandas as pd

df = pd.read_csv("/content/zotutab_decontam_taxa_nb.csv")
df = pd.read_csv("/content/zotutab_decontam.relative_abundance.csv")

# Skip the ANATOMY row if present
data = df.iloc[1:].copy()

def split_taxonomy(t):
    if pd.isna(t):
        return ["Unknown"]*7
    parts = t.split(";")
    ranks = {"k__":"","p__":"","c__":"","o__":"","f__":"","g__":"","s__":""}
    for p in parts:
        for key in ranks:
            if p.strip().startswith(key):
                ranks[key] = p.replace(key,"").strip()
    return [
        ranks["k__"],
        ranks["p__"],
        ranks["c__"],
        ranks["o__"],
        ranks["f__"],
        ranks["g__"],
        ranks["s__"]
    ]

tax_cols = data["Taxonomy"].apply(split_taxonomy)
tax_df = pd.DataFrame(tax_cols.tolist(),
                      columns=["Domain","Phylum","Class","Order","Family","Genus","Species"])

# Combine
df_out = pd.concat([data.drop(columns=["Taxonomy"]), tax_df], axis=1)

df_out.to_csv("//content/zotutab_decontam.relab_taxasplit.csv", index=False)
df_out.to_csv("zotutab_decontam_RelAbund_Final.csv", index=False)
print("Saved")


In [ ]:
# ============================
# STANDARDISE PHYLUM NAMES all verified!
# ============================
rename_map = {
    'actinobacteria':   'actinomycetota',
    'firmicutes':       'bacillota',
    'bacteroidetes':    'bacteroidota',
    'chloroflexi':      'chloroflexota',
    'cyanobacteria':    'cyanobacteriota',
    'proteobacteria':   'pseudomonadota',
    'pseudomonadot':    'pseudomonadota',
    'planctomycetes':   'planctomycetota',
    'gemmatimonadetes': 'gemmatimonadota',
    'verrucomicrobia':  'verrucomicrobiota',
    'acidobacteria':    'acidobacteriota',
    'nitrospirae':      'nitrospirota',
    'chlamydiae':       'chlamydiota',       # add
    'elusimicrobia':    'elusimicrobiota',   # add
    'spirochaetes':     'spirochaetota',     # add
    'tenericutes':      'mycoplasmatota',    # add
}

In [3]:
import pandas as pd

# ============================
# LOAD FILE
# ============================
path = "/content/zotutab_decontam_unstandardized_naming.csv"
df   = pd.read_csv(path, header=None)

print(f"Original shape: {df.shape}")
print(f"First few column names: {df.iloc[1, :5].tolist()}")

# ============================
# IDENTIFY STRUCTURE
# Row 0 = file header info
# Row 1 = sample/taxonomy column names
# Row 2+ = data
# ============================
header_row  = df.iloc[1]
data        = df.iloc[2:].copy()
data.columns = header_row.values
data.reset_index(drop=True, inplace=True)

# ============================
# COMPLETE RENAME MAP
# ============================
rename_map = {
    'actinobacteria':        'actinomycetota',
    'firmicutes':            'bacillota',
    'bacteroidetes':         'bacteroidota',
    'chloroflexi':           'chloroflexota',
    'cyanobacteria':         'cyanobacteriota',
    'proteobacteria':        'pseudomonadota',
    'pseudomonadot':         'pseudomonadota',
    'planctomycetes':        'planctomycetota',
    'gemmatimonadetes':      'gemmatimonadota',
    'verrucomicrobia':       'verrucomicrobiota',
    'acidobacteria':         'acidobacteriota',
    'nitrospirae':           'nitrospirota',
    'chlamydiae':            'chlamydiota',
    'elusimicrobia':         'elusimicrobiota',
    'spirochaetes':          'spirochaetota',
    'tenericutes':           'mycoplasmatota',
    'fusobacteria':          'fusobacteriota',
    'thermodesulfobacteria': 'thermodesulfobacteriota',
    'deinococcus-thermus':   'deinococcota',
    'armatimonadetes':       'armatimonadota',
    'thermotogae':           'thermotogota',
}

# ============================
# APPLY RENAME TO PHYLUM COLUMN
# ============================
# check phylum column exists
if "Phylum" not in data.columns:
    print("ERROR: Phylum column not found")
    print(f"Available columns: {data.columns.tolist()[:10]}")
else:
    # normalise to lowercase, strip whitespace, then remap
    original_phyla  = data["Phylum"].copy()
    data["Phylum"]  = (
        data["Phylum"]
        .str.strip()
        .str.lower()
        .replace(rename_map)
    )

    # ============================
    # AUDIT — show what changed
    # ============================
    changed = original_phyla != data["Phylum"]
    changes = pd.DataFrame({
        "Original": original_phyla[changed],
        "Updated":  data["Phylum"][changed]
    }).drop_duplicates()

    print(f"\nPhylum names updated: {changed.sum()} rows")
    print(f"Unique changes made:  {len(changes)}")
    print()
    print(changes.to_string(index=False))

    # ============================
    # VERIFY NO OLD NAMES REMAIN
    # ============================
    old_names = list(rename_map.keys())
    remaining = data["Phylum"].isin(old_names).sum()
    print(f"\nOld names still present: {remaining} "
          f"(should be 0)")

    # ============================
    # CHECK UNIQUE PHYLA AFTER UPDATE
    # ============================
    print(f"\nUnique phyla after update: "
          f"{data['Phylum'].nunique()}")
    print("\nAll unique phyla:")
    for p in sorted(data["Phylum"].dropna().unique()):
        print(f"  {p}")
        #print(len("Phylum"))

    # ============================
    # SAVE NEW CSV
    # preserving original file structure
    # ============================
    # rebuild full dataframe with original top rows
    out = pd.concat([df.iloc[:2],
                     pd.DataFrame([data.columns.tolist()
                                   if False else None])],
                    ignore_index=True)

    # simpler approach — just save data rows with header
    output_path = "/content/zotutab_decontam_Final_standardized.csv"
    data.to_csv(output_path, index=False)

    print(f"\nSaved: {output_path}")
    print(f"Shape: {data.shape}")

Original shape: (6200, 70)
First few column names: ['Zotu', 'MF.4B', 'MF.5B', 'MF.6B', 'MF.7B']

Phylum names updated: 991 rows
Unique changes made:  18

        Original           Updated
   bacteroidetes      bacteroidota
  proteobacteria    pseudomonadota
   acidobacteria   acidobacteriota
  actinobacteria    actinomycetota
      firmicutes         bacillota
             NaN               NaN
     chloroflexi     chloroflexota
   cyanobacteria   cyanobacteriota
 verrucomicrobia verrucomicrobiota
   pseudomonadot    pseudomonadota
  planctomycetes   planctomycetota
gemmatimonadetes   gemmatimonadota
   elusimicrobia   elusimicrobiota
     nitrospirae      nitrospirota
    fusobacteria    fusobacteriota
    spirochaetes     spirochaetota
     tenericutes    mycoplasmatota
      chlamydiae       chlamydiota

Old names still present: 0 (should be 0)

Unique phyla after update: 49

All unique phyla:
  abditibacteriota
  acidobacteriota
  actinomycetota
  armatimonadota
  atribacterota
  

In [4]:
import pandas as pd

# ============================
# LOAD STANDARDIZED FILE
# ============================
path = "/content/zotutab_decontam_Final_standardized.csv"
df   = pd.read_csv(path)

print(f"Total ZOTUs: {len(df)}")

# ============================
# NORMALISE DOMAIN
# ============================
df["Domain_norm"] = df["Domain"].str.strip().str.lower()
df["Phylum_norm"] = df["Phylum"].str.strip().str.lower()

# ============================
# IDENTIFY NON-PROKARYOTES
# anything that is not bacteria or archaea
# ============================
prokaryote_domains = ["bacteria", "archaea"]
non_prok = df[~df["Domain_norm"].isin(prokaryote_domains)].copy()

print(f"\nNon-prokaryotic ZOTUs: {len(non_prok)}")

# ============================
# PRINT ALL UNIQUE DOMAIN AND
# PHYLUM NAMES FOR NON-PROKARYOTES
# ============================
print("\n" + "="*50)
print("UNIQUE DOMAINS (non-prokaryotic):")
print("="*50)
for d in sorted(non_prok["Domain"].dropna().unique()):
    count = (non_prok["Domain"] == d).sum()
    print(f"  {d:<35} n={count}")

print("\n" + "="*50)
print("UNIQUE PHYLA (non-prokaryotic):")
print("="*50)
for p in sorted(non_prok["Phylum"].dropna().unique()):
    count = (non_prok["Phylum"] == p).sum()
    print(f"  {p:<35} n={count}")

print("\n" + "="*50)
print("UNCLASSIFIED AT PHYLUM LEVEL:")
print("="*50)
unclass = non_prok["Phylum"].isna().sum()
print(f"  No phylum assignment: {unclass}")

print("\n" + "="*50)
print("FULL BREAKDOWN — DOMAIN x PHYLUM:")
print("="*50)
breakdown = non_prok.groupby(
    ["Domain", "Phylum"], dropna=False).size().reset_index(
    name="count")
print(breakdown.to_string(index=False))

Total ZOTUs: 6198

Non-prokaryotic ZOTUs: 302

UNIQUE DOMAINS (non-prokaryotic):
  eukaryota                           n=34
  fungi                               n=1
  metazoa                             n=3
  viridiplantae                       n=14

UNIQUE PHYLA (non-prokaryotic):
  bacillariophyta                     n=1
  basidiomycota                       n=1
  chlorophyta                         n=5
  chordata                            n=3
  ciliophora                          n=2
  discosea                            n=6
  eukaryota                           n=18
  eustigmatophyceae                   n=1
  oomycota                            n=1
  phaeophyceae                        n=1
  streptophyta                        n=9
  tubulinea                           n=4

UNCLASSIFIED AT PHYLUM LEVEL:
  No phylum assignment: 250

FULL BREAKDOWN — DOMAIN x PHYLUM:
       Domain            Phylum  count
    eukaryota   bacillariophyta      1
    eukaryota        ciliophora      2


In [5]:
import pandas as pd

# ============================
# LOAD STANDARDIZED FILE
# ============================
path = "/content/zotutab_decontam_Final_standardized.csv"
df   = pd.read_csv(path)

print(f"Total ZOTUs before removal: {len(df)}")

# ============================
# NORMALISE
# ============================
df["Domain_norm"] = df["Domain"].str.strip().str.lower()
df["Phylum_norm"] = df["Phylum"].str.strip().str.lower()

# ============================
# EUKARYOTE IDENTIFIERS
# confirmed from output above
# ============================
eukaryote_domains = [
    "eukaryota", "fungi", "metazoa", "viridiplantae"
]

eukaryote_phyla = [
    "bacillariophyta", "basidiomycota", "chlorophyta",
    "chordata", "ciliophora", "discosea", "eukaryota",
    "eustigmatophyceae", "oomycota", "phaeophyceae",
    "streptophyta", "tubulinea",
]

# ============================
# FLAG EUKARYOTES
# remove by domain OR phylum OR unclassified non-prokaryote
# ============================
euk_by_domain = df["Domain_norm"].isin(eukaryote_domains)
euk_by_phylum = df["Phylum_norm"].isin(eukaryote_phyla)

# unclassified non-prokaryotes — NaN phylum and non-prokaryote domain
euk_unclassified = (
    df["Phylum_norm"].isna() &
    ~df["Domain_norm"].isin(["bacteria", "archaea"])
)

euk_mask = euk_by_domain | euk_by_phylum | euk_unclassified

# ============================
# AUDIT
# ============================
print(f"\nRemoval audit:")
print(f"  Flagged by domain:              {euk_by_domain.sum()}")
print(f"  Flagged by phylum:              {euk_by_phylum.sum()}")
print(f"  Flagged as unclassified non-prok: {euk_unclassified.sum()}")
print(f"  Total flagged:                  {euk_mask.sum()}")
print(f"  Expected removal:               302")
print(f"  Match: {euk_mask.sum() == 302}")

# ============================
# REMOVE AND CLEAN
# ============================
df_clean = df[~euk_mask].copy()
df_clean = df_clean.drop(columns=["Domain_norm", "Phylum_norm"])

# ============================
# VERIFY
# ============================
remaining_euk = df_clean["Domain"].str.strip().str.lower().isin(
    eukaryote_domains).sum()
remaining_prok = len(df_clean)

print(f"\nVerification:")
print(f"  ZOTUs remaining:              {remaining_prok}")
print(f"  Expected prokaryotic ZOTUs:   5896")
print(f"  Match: {remaining_prok == 5896}")
print(f"  Eukaryotic domains remaining: {remaining_euk} (should be 0)")

print(f"\nDomain breakdown after removal:")
print(df_clean["Domain"].str.lower().str.strip()
      .value_counts().to_string())

print(f"\nUnique phyla after removal: "
      f"{df_clean['Phylum'].nunique()}")

# ============================
# SAVE
# ============================
output_path = "/content/zotutab_decontam_Final_prokaryotes.csv"
df_clean.to_csv(output_path, index=False)
print(f"\nSaved: {output_path}")
print(f"Final shape: {df_clean.shape}")

Total ZOTUs before removal: 6198

Removal audit:
  Flagged by domain:              52
  Flagged by phylum:              52
  Flagged as unclassified non-prok: 250
  Total flagged:                  302
  Expected removal:               302
  Match: True

Verification:
  ZOTUs remaining:              5896
  Expected prokaryotic ZOTUs:   5896
  Match: True
  Eukaryotic domains remaining: 0 (should be 0)

Domain breakdown after removal:
Domain
bacteria    5547
archaea      349

Unique phyla after removal: 37

Saved: /content/zotutab_decontam_Final_prokaryotes.csv
Final shape: (5896, 70)


In [6]:
import pandas as pd

# ============================
# LOAD FINAL PROKARYOTE FILE
# ============================
path = "/content/zotutab_decontam_Final_prokaryotes.csv"
df   = pd.read_csv(path)

print("=" * 60)
print("QUALITY CHECK — zotutab_decontam_Final_prokaryotes.csv")
print("=" * 60)

# ============================
# RENAME MAP — all old names
# that should NOT appear
# ============================
old_names = [
    'actinobacteria', 'firmicutes', 'bacteroidetes',
    'chloroflexi', 'cyanobacteria', 'proteobacteria',
    'pseudomonadot', 'planctomycetes', 'gemmatimonadetes',
    'verrucomicrobia', 'acidobacteria', 'nitrospirae',
    'chlamydiae', 'elusimicrobia', 'spirochaetes',
    'tenericutes', 'fusobacteria', 'thermodesulfobacteria',
    'deinococcus-thermus', 'armatimonadetes', 'thermotogae',
]

eukaryote_domains = [
    'eukaryota', 'fungi', 'metazoa', 'viridiplantae'
]

eukaryote_phyla = [
    'bacillariophyta', 'basidiomycota', 'chlorophyta',
    'chordata', 'ciliophora', 'discosea', 'eukaryota',
    'eustigmatophyceae', 'oomycota', 'phaeophyceae',
    'streptophyta', 'tubulinea',
]

# ============================
# NORMALISE FOR CHECKING
# ============================
df["Domain_norm"] = df["Domain"].str.strip().str.lower()
df["Phylum_norm"] = df["Phylum"].str.strip().str.lower()

# ============================
# CHECK 1 — TOTAL ZOTU COUNT
# expected: 5896
# ============================
print(f"\n{'CHECK 1 — TOTAL ZOTU COUNT':}")
print(f"  Total ZOTUs: {len(df)}")
print(f"  Expected:    5896")
print(f"  {'PASS' if len(df) == 5896 else 'FAIL'}")

# ============================
# CHECK 2 — NO BLANK SAMPLES
# ============================
print(f"\nCHECK 2 — NO BLANK SAMPLES")
blank_cols = [c for c in df.columns if "Blank" in str(c)
              or "blank" in str(c)]
print(f"  Blank columns found: {len(blank_cols)}")
if blank_cols:
    print(f"  FAIL — blanks present: {blank_cols}")
else:
    print(f"  PASS — no blank columns")

# ============================
# CHECK 3 — NO EUKARYOTES BY DOMAIN
# ============================
print(f"\nCHECK 3 — NO EUKARYOTIC DOMAINS")
euk_domain = df["Domain_norm"].isin(eukaryote_domains).sum()
print(f"  Eukaryotic domain rows: {euk_domain}")
print(f"  {'PASS' if euk_domain == 0 else 'FAIL'}")
if euk_domain > 0:
    print(f"  Domains found: "
          f"{df.loc[df['Domain_norm'].isin(eukaryote_domains), 'Domain'].unique()}")

# ============================
# CHECK 4 — NO EUKARYOTES BY PHYLUM
# ============================
print(f"\nCHECK 4 — NO EUKARYOTIC PHYLA")
euk_phylum = df["Phylum_norm"].isin(eukaryote_phyla).sum()
print(f"  Eukaryotic phylum rows: {euk_phylum}")
print(f"  {'PASS' if euk_phylum == 0 else 'FAIL'}")
if euk_phylum > 0:
    print(f"  Phyla found: "
          f"{df.loc[df['Phylum_norm'].isin(eukaryote_phyla), 'Phylum'].unique()}")

# ============================
# CHECK 5 — NO OLD PHYLUM NAMES
# ============================
print(f"\nCHECK 5 — NO OLD PHYLUM NAMES (rename map applied)")
old_found = df["Phylum_norm"].isin(old_names)
print(f"  Old names found: {old_found.sum()}")
print(f"  {'PASS' if old_found.sum() == 0 else 'FAIL'}")
if old_found.sum() > 0:
    print(f"  Old names still present:")
    for n in df.loc[old_found, "Phylum"].unique():
        print(f"    {n}")

# ============================
# CHECK 6 — NO UNCLASSIFIED PHYLA
# ============================
print(f"\nCHECK 6 — NO UNCLASSIFIED PHYLA")
unclass_phylum = df["Phylum_norm"].isna().sum()
unclass_str    = df["Phylum_norm"].str.startswith(
    "unclassified", na=False).sum()
print(f"  NaN phylum:            {unclass_phylum}")
print(f"  'unclassified' string: {unclass_str}")
total_unclass = unclass_phylum + unclass_str
print(f"  Total unclassified:    {total_unclass}")
print(f"  {'PASS' if total_unclass == 0 else 'FAIL'}")

# ============================
# CHECK 7 — ONLY BACTERIA AND ARCHAEA
# ============================
print(f"\nCHECK 7 — ONLY BACTERIA AND ARCHAEA")
domain_counts = df["Domain_norm"].value_counts()
print(f"  Domain breakdown:")
for d, n in domain_counts.items():
    print(f"    {d:<20} n={n}")
only_prok = set(domain_counts.index) <= {"bacteria", "archaea"}
print(f"  {'PASS' if only_prok else 'FAIL'}")

# ============================
# CHECK 8 — BACTERIA AND ARCHAEA COUNTS
# ============================
print(f"\nCHECK 8 — BACTERIA AND ARCHAEA COUNTS")
n_bact = (df["Domain_norm"] == "bacteria").sum()
n_arch = (df["Domain_norm"] == "archaea").sum()
print(f"  Bacteria: {n_bact}  (expected 5547)")
print(f"  Archaea:  {n_arch}  (expected 349)")
print(f"  Bacteria {'PASS' if n_bact == 5547 else 'FAIL'}")
print(f"  Archaea  {'PASS' if n_arch == 349  else 'FAIL'}")

# ============================
# CHECK 9 — UNIQUE PHYLA
# ============================
print(f"\nCHECK 9 — UNIQUE PHYLA COUNT")
n_phyla = df["Phylum_norm"].nunique()
print(f"  Unique phyla: {n_phyla}  (expected 36)")
print(f"  {'PASS' if n_phyla == 36 else 'FAIL'}")

print(f"\nAll unique phyla:")
for p in sorted(df["Phylum_norm"].dropna().unique()):
    n = (df["Phylum_norm"] == p).sum()
    print(f"  {p:<40} n={n}")

# ============================
# SUMMARY
# ============================
checks = {
    "Total ZOTUs = 5896":        len(df) == 5896,
    "No blank columns":          len(blank_cols) == 0,
    "No eukaryotic domains":     euk_domain == 0,
    "No eukaryotic phyla":       euk_phylum == 0,
    "No old phylum names":       old_found.sum() == 0,
    "No unclassified phyla":     total_unclass == 0,
    "Only bacteria and archaea": only_prok,
    "Bacteria count = 5547":     n_bact == 5547,
    "Archaea count = 349":       n_arch == 349,
    "Unique phyla = 36":         n_phyla == 36,
}

print("\n" + "=" * 60)
print("SUMMARY")
print("=" * 60)
all_pass = True
for check, result in checks.items():
    status = "PASS" if result else "FAIL"
    if not result:
        all_pass = False
    print(f"  {status}  {check}")

print()
if all_pass:
    print("ALL CHECKS PASSED — file is clean and ready to use")
else:
    print("SOME CHECKS FAILED — review output above before proceeding")

# clean up normalisation columns
df.drop(columns=["Domain_norm", "Phylum_norm"], inplace=True)

QUALITY CHECK — zotutab_decontam_Final_prokaryotes.csv

CHECK 1 — TOTAL ZOTU COUNT
  Total ZOTUs: 5896
  Expected:    5896
  PASS

CHECK 2 — NO BLANK SAMPLES
  Blank columns found: 0
  PASS — no blank columns

CHECK 3 — NO EUKARYOTIC DOMAINS
  Eukaryotic domain rows: 0
  PASS

CHECK 4 — NO EUKARYOTIC PHYLA
  Eukaryotic phylum rows: 0
  PASS

CHECK 5 — NO OLD PHYLUM NAMES (rename map applied)
  Old names found: 0
  PASS

CHECK 6 — NO UNCLASSIFIED PHYLA
  NaN phylum:            0
  'unclassified' string: 0
  Total unclassified:    0
  PASS

CHECK 7 — ONLY BACTERIA AND ARCHAEA
  Domain breakdown:
    bacteria             n=5547
    archaea              n=349
  PASS

CHECK 8 — BACTERIA AND ARCHAEA COUNTS
  Bacteria: 5547  (expected 5547)
  Archaea:  349  (expected 349)
  Bacteria PASS
  Archaea  PASS

CHECK 9 — UNIQUE PHYLA COUNT
  Unique phyla: 37  (expected 36)
  FAIL

All unique phyla:
  abditibacteriota                         n=6
  acidobacteriota                          n=255
  acti

In [7]:
import pandas as pd

# ============================
# LOAD FILE
# ============================
path = "/content/zotutab_decontam_Final_prokaryotes.csv"
df   = pd.read_csv(path)

print(f"ZOTUs before fix: {len(df)}")

# ============================
# INSPECT THE 4 PROBLEM ZOTUs
# ============================
df["Phylum_norm"] = df["Phylum"].str.strip().str.lower()

problem = df[df["Phylum_norm"] == "bacteria"]
print(f"\nZOTUs with Phylum = 'bacteria': {len(problem)}")
print(problem[["Domain", "Phylum", "Class",
               "Order", "Family"]].to_string())

# ============================
# REMOVE THEM
# phylum = 'bacteria' is not a valid phylum
# these are unclassified at phylum level
# ============================
df_clean = df[df["Phylum_norm"] != "bacteria"].copy()
df_clean = df_clean.drop(columns=["Phylum_norm"])

print(f"\nZOTUs after fix:  {len(df_clean)}")
print(f"Removed:          {len(df) - len(df_clean)}")
print(f"Expected total:   5892")
print(f"{'PASS' if len(df_clean) == 5892 else 'FAIL'}")

# ============================
# VERIFY PHYLA COUNT
# ============================
n_phyla = df_clean["Phylum"].str.strip().str.lower().nunique()
print(f"\nUnique phyla after fix: {n_phyla}  (expected 36)")
print(f"{'PASS' if n_phyla == 36 else 'FAIL'}")

# ============================
# SAVE
# ============================
output_path = "/content/zotutab_decontam_Final.csv"
df_clean.to_csv(output_path, index=False)
print(f"\nSaved: {output_path}")

ZOTUs before fix: 5896

ZOTUs with Phylum = 'bacteria': 4
        Domain    Phylum                Class                Order                Family
1660  bacteria  bacteria  deltaproteobacteria  deltaproteobacteria  dissulfurirhabdaceae
2353  bacteria  bacteria  deltaproteobacteria  deltaproteobacteria  dissulfurirhabdaceae
4838  bacteria  bacteria  deltaproteobacteria  deltaproteobacteria  dissulfurirhabdaceae
5289  bacteria  bacteria  deltaproteobacteria  deltaproteobacteria  dissulfurirhabdaceae

ZOTUs after fix:  5892
Removed:          4
Expected total:   5892
PASS

Unique phyla after fix: 36  (expected 36)
PASS

Saved: /content/zotutab_decontam_Final.csv


In [8]:
import pandas as pd

# ============================
# LOAD CLEANED PROKARYOTE FILE
# ============================
path = "/content/zotutab_decontam_Final.csv"
df   = pd.read_csv(path)

print(f"Loaded: {df.shape[0]} ZOTUs x {df.shape[1]} columns")

# ============================
# IDENTIFY SAMPLE COLUMNS
# everything that is not a taxonomy column
# ============================
tax_cols = ["Domain", "Phylum", "Class", "Order",
            "Family", "Genus", "Species"]

# get the ZOTU ID column — first column
zotu_col = df.columns[0]

sample_cols = [c for c in df.columns
               if c not in tax_cols
               and c != zotu_col
               and "Blank" not in str(c)]

print(f"ZOTU ID column:    {zotu_col}")
print(f"Taxonomy columns:  {len(tax_cols)}")
print(f"Sample columns:    {len(sample_cols)}")
print(f"First 3 samples:   {sample_cols[:3]}")

# ============================
# CONVERT TO NUMERIC
# ============================
df[sample_cols] = df[sample_cols].apply(
    pd.to_numeric, errors="coerce").fillna(0)

# ============================
# CALCULATE RELATIVE ABUNDANCE
# for each sample divide each ZOTU count
# by the total reads in that sample
# multiply by 100 to get percentage
# ============================
col_sums = df[sample_cols].sum(axis=0)

print(f"\nRead depth per sample (before conversion):")
print(f"  Min:  {col_sums.min():.0f}")
print(f"  Max:  {col_sums.max():.0f}")
print(f"  Mean: {col_sums.mean():.0f}")

# divide each column by its sum and multiply by 100
rel_abund = df[sample_cols].div(col_sums, axis=1) * 100

# ============================
# VERIFY — each sample should sum to 100%
# ============================
col_sums_check = rel_abund.sum(axis=0)
print(f"\nVerification — sample column sums after conversion:")
print(f"  Min:  {col_sums_check.min():.4f}%  (should be 100)")
print(f"  Max:  {col_sums_check.max():.4f}%  (should be 100)")
print(f"  Mean: {col_sums_check.mean():.4f}%  (should be 100)")
all_100 = (col_sums_check.round(6) == 100).all()
print(f"  All samples sum to 100%: {all_100}")

# ============================
# BUILD OUTPUT DATAFRAME
# keep ZOTU ID and taxonomy columns
# replace count columns with relative abundance
# ============================
df_rel = df[[zotu_col] + tax_cols].copy()
df_rel = pd.concat([df_rel, rel_abund], axis=1)

print(f"\nOutput shape: {df_rel.shape}")
print(f"Columns: {zotu_col}, {tax_cols}, + {len(sample_cols)} sample columns")

# ============================
# SPOT CHECK — one sample
# ============================
test_sample = sample_cols[0]
print(f"\nSpot check — {test_sample}:")
print(f"  Original total reads: {col_sums[test_sample]:.0f}")
print(f"  Rel abund sum:        {rel_abund[test_sample].sum():.4f}%")
print(f"  Top 3 ZOTUs by abundance:")
top3 = rel_abund[test_sample].nlargest(3)
for zotu, val in top3.items():
    print(f"    {zotu}: {val:.4f}%")

# ============================
# SAVE
# ============================
output_path = "/content/zotutab_decontam_RelAbund_Final.csv"
df_rel.to_csv(output_path, index=False)
print(f"\nSaved: {output_path}")
print(f"Final shape: {df_rel.shape}")

Loaded: 5892 ZOTUs x 70 columns
ZOTU ID column:    Zotu
Taxonomy columns:  7
Sample columns:    62
First 3 samples:   ['MF.4B', 'MF.5B', 'MF.6B']

Read depth per sample (before conversion):
  Min:  24903
  Max:  32634
  Mean: 30775

Verification — sample column sums after conversion:
  Min:  100.0000%  (should be 100)
  Max:  100.0000%  (should be 100)
  Mean: 100.0000%  (should be 100)
  All samples sum to 100%: True

Output shape: (5892, 70)
Columns: Zotu, ['Domain', 'Phylum', 'Class', 'Order', 'Family', 'Genus', 'Species'], + 62 sample columns

Spot check — MF.4B:
  Original total reads: 30958
  Rel abund sum:        100.0000%
  Top 3 ZOTUs by abundance:
    2: 19.0742%
    1: 13.4085%
    7: 3.4918%

Saved: /content/zotutab_decontam_RelAbund_Final.csv
Final shape: (5892, 70)
